# Fase 2 — Comprensión de los Datos
## Sección 3: Calidad de Datos — Diagnóstico Inicial

**Notebook:** `notebooks/02_EDA.ipynb`
**Responsable:** Sofía | **Apoyo:** Steve
**Objetivo:** Evaluar la calidad de los 16 datasets (Grupo A y B): completitud, duplicados, nulos y patrones de datos faltantes.

In [ ]:
%pip install seaborn

## Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RAW = os.path.join("..", "data", "raw")
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

print(f"Raw path: {os.path.abspath(RAW)}")
print(f"Exists: {os.path.isdir(RAW)}")


---
## Celdas 26–30: Estadísticas de calidad por dataset

### Cargar todos los datasets y calcular métricas de calidad

In [ ]:
A_FILES = [
    ("A1", "A1_colombia_housing_properties.csv"),
    ("A2", "A2_fincaraiz_colombia.csv"),
    ("A3", "A3_colombia_house_prediction.csv"),
    ("A4", "A4_real_estate_bogota.csv"),
    ("A5", "A5_medellin_properties_2023.csv"),
    ("A6", "A6_real_estate_bogota_2023.csv"),
    ("A7", "A7_fincaraiz_villavicencio_scraping.csv"),
    ("A8", "A8_carac_pre_viv_nueva.csv"),
]

B_FILES = [
    ("B1", "B1_indices_precios_vivienda.csv"),
    ("B2", "B2_tasa_hipotecaria_semanal.csv"),
    ("B3", "B3_salario_minimo_historico.csv"),
    ("B4", "B4_ipc_colombia_anual.csv"),
    ("B5", "B5_geih_empleo_colombia.csv"),
    ("B6", "B6_qcon_confianza_constructora.csv"),
    ("B7", "B7_qcon_licencias_construccion.csv"),
    ("B8", "B8_geo_estados_localidades.csv"),
]

ALL_FILES = A_FILES + B_FILES

datasets = {}
for fid, fname in ALL_FILES:
    fpath = os.path.join(RAW, fname)
    try:
        df = pd.read_csv(fpath, encoding="utf-8-sig", low_memory=False)
        datasets[fid] = df
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"[OK] {fid:2s} {fname:45s} {df.shape[0]:>8,} filas x {df.shape[1]:>3} cols  {size_mb:>8.1f} MB")
    except Exception as e:
        print(f"[ERR] {fid:2s} {fname}: {e}")
        datasets[fid] = None


### Tabla resumen de calidad — Grupo A

In [ ]:
rows = []
for fid, _ in A_FILES:
    df = datasets[fid]
    if df is None:
        continue
    n_rows, n_cols = df.shape
    size_mb = df.memory_usage(deep=True).sum() / 1e6
    pct_nulos_total = df.isnull().sum().sum() / (n_rows * n_cols) * 100
    duplicados = df.duplicated().sum()
    pct_precio_nulo = None
    pct_area_nula = None

    for precio_col in ["price", "precio", "valor", "precios", "Valor", "precio_cop", "Precio"]:
        if precio_col in df.columns:
            pct_precio_nulo = round(df[precio_col].isnull().mean() * 100, 2)
            break

    for area_col in ["area", "Area", "Area Construida", "area_m2"]:
        if area_col in df.columns:
            pct_area_nula = round(df[area_col].isnull().mean() * 100, 2)
            break

    rows.append({
        "Dataset": fid,
        "Filas": n_rows,
        "Columnas": n_cols,
        "Memoria (MB)": round(size_mb, 2),
        "% Nulos total": round(pct_nulos_total, 2),
        "Duplicados": duplicados,
        "% Precio nulo": pct_precio_nulo if pct_precio_nulo is not None else "N/A",
        "% Area nula": pct_area_nula if pct_area_nula is not None else "N/A",
    })

df_quality_a = pd.DataFrame(rows)
display(df_quality_a)


### Tabla resumen de calidad — Grupo B

In [ ]:
rows_b = []
for fid, _ in B_FILES:
    df = datasets[fid]
    if df is None:
        continue
    n_rows, n_cols = df.shape
    size_mb = df.memory_usage(deep=True).sum() / 1e6
    pct_nulos_total = df.isnull().sum().sum() / (n_rows * n_cols) * 100
    duplicados = df.duplicated().sum()

    rows_b.append({
        "Dataset": fid,
        "Filas": n_rows,
        "Columnas": n_cols,
        "Memoria (MB)": round(size_mb, 2),
        "% Nulos total": round(pct_nulos_total, 2),
        "Duplicados": duplicados,
    })

df_quality_b = pd.DataFrame(rows_b)
display(df_quality_b)


### Resumen consolidado de calidad

In [ ]:
print("===== RESUMEN CALIDAD GRUPO A =====")
display(df_quality_a)

print("\n===== RESUMEN CALIDAD GRUPO B =====")
display(df_quality_b)

df_quality_a.to_csv(os.path.join(OUT, "calidad_grupo_a.csv"), index=False)
df_quality_b.to_csv(os.path.join(OUT, "calidad_grupo_b.csv"), index=False)
print(f"\nReportes guardados en {OUT}/")


---
## Celdas 31–35: Análisis de nulos

### Mapa de calor de valores nulos (muestra 5000 filas por dataset)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (fid, _) in enumerate(A_FILES):
    df = datasets[fid]
    if df is None:
        axes[i].text(0.5, 0.5, f"{fid}: error carga", ha="center", va="center")
        continue
    sample = df.sample(min(5000, len(df)), random_state=42)
    sns.heatmap(sample.isnull(), cbar=False, yticklabels=False, ax=axes[i], cmap="viridis")
    axes[i].set_title(f"{fid} — {df.shape[0]:,} filas")
    axes[i].set_xlabel("")

plt.tight_layout()
plt.savefig(os.path.join(OUT, "mapa_calor_nulos_grupo_a.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Mapa de calor guardado.")


### Porcentaje de nulos por columna — Grupo A

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (fid, _) in enumerate(A_FILES):
    df = datasets[fid]
    if df is None:
        continue
    pct_nulos = df.isnull().mean().sort_values(ascending=False) * 100
    pct_nulos = pct_nulos[pct_nulos > 0]
    if len(pct_nulos) == 0:
        axes[i].text(0.5, 0.5, "Sin nulos", ha="center", va="center")
        continue
    axes[i].barh(range(len(pct_nulos)), pct_nulos.values)
    axes[i].set_yticks(range(len(pct_nulos)))
    axes[i].set_yticklabels(pct_nulos.index, fontsize=8)
    axes[i].set_xlabel("% Nulos")
    axes[i].set_title(f"{fid} — {len(pct_nulos)} cols con nulos")

plt.tight_layout()
plt.savefig(os.path.join(OUT, "pct_nulos_por_columna_grupo_a.png"), dpi=150, bbox_inches="tight")
plt.show()


### Análisis de nulos en área por dataset

In [ ]:
print("===== NULOS EN ÁREA POR CIUDAD =====\n")

area_cols_map = {
    "A1": "area",
    "A2": "Area Construida",
    "A3": "area",
    "A4": "Area",
    "A5": "area",
    "A6": "Area",
    "A7": "area_m2",
    "A8": "area",
}

city_cols_map = {
    "A1": "l3",
    "A2": "Ciudad",
    "A3": None,
    "A4": None,
    "A5": "neighbourhood",
    "A6": None,
    "A7": "ciudad",
    "A8": None,
}

for fid, _ in A_FILES:
    df = datasets[fid]
    if df is None:
        continue
    area_col = area_cols_map.get(fid)
    if area_col not in df.columns:
        print(f"{fid}: columna de area no encontrada")
        continue

    pct_nulo_area = df[area_col].isnull().mean() * 100

    city_col = city_cols_map.get(fid)
    if city_col and city_col in df.columns:
        print(f"\n{fid} — {pct_nulo_area:.1f}% area nulo (por ciudad):")
        by_city = df.groupby(city_col)[area_col].apply(lambda x: x.isnull().mean() * 100)
        for city, pct in by_city.sort_values(ascending=False).head(10).items():
            bar = "#" * int(pct / 2)
            print(f"  {str(city):30s} {pct:5.1f}% {bar}")
    else:
        print(f"{fid}: {pct_nulo_area:.1f}% area nulo (sin columna ciudad)")


### Análisis de nulos en precio por dataset

In [ ]:
print("===== NULOS EN PRECIO POR CIUDAD =====\n")

price_cols_map = {
    "A1": "price",
    "A2": "Precio",
    "A3": "valor",
    "A4": "Valor",
    "A5": "price",
    "A6": "precio",
    "A7": "precio_cop",
    "A8": "precios",
}

for fid, _ in A_FILES:
    df = datasets[fid]
    if df is None:
        continue
    price_col = price_cols_map.get(fid)
    if price_col not in df.columns:
        print(f"{fid}: columna precio no encontrada")
        continue

    pct = df[price_col].isnull().mean() * 100
    city_col = city_cols_map.get(fid)
    if city_col and city_col in df.columns:
        print(f"{fid}: {pct:.1f}% precio nulo (por ciudad):")
        by_city = df.groupby(city_col)[price_col].apply(lambda x: x.isnull().mean() * 100)
        for city, v in by_city.sort_values(ascending=False).head(5).items():
            print(f"    {str(city):30s} {v:5.1f}%")
    else:
        print(f"{fid}: {pct:.1f}% precio nulo")


### Guardar reporte de nulos completo

In [ ]:
print("===== REPORTE DE NULOS COMPLETO =====\n")

nulos_report = []
for fid, _ in ALL_FILES:
    df = datasets[fid]
    if df is None:
        continue
    for col in df.columns:
        n_nulos = df[col].isnull().sum()
        if n_nulos > 0:
            nulos_report.append({
                "Dataset": fid,
                "Columna": col,
                "Nulos": n_nulos,
                "Total filas": len(df),
                "% Nulos": round(n_nulos / len(df) * 100, 2),
                "Dtype": str(df[col].dtype),
            })

df_nulos = pd.DataFrame(nulos_report)
df_nulos = df_nulos.sort_values("% Nulos", ascending=False)
display(df_nulos.head(30))
print(f"\nTotal columnas con nulos: {len(df_nulos)}")

df_nulos.to_csv(os.path.join(OUT, "reporte_nulos_completo.csv"), index=False)
print(f"Reporte guardado: {OUT}/reporte_nulos_completo.csv")


### Patrones de nulos por dataset — tabla pivot

In [ ]:
nulos_pivot = df_nulos.pivot_table(
    index="Dataset",
    columns="Columna",
    values="% Nulos",
    aggfunc="first"
)
print("===== MATRIZ DE NULOS (% por Dataset/Columna) =====\n")
display(nulos_pivot.fillna(0).style.background_gradient(cmap="Reds", axis=None))


---
## Resumen de la Sección 3

- Se cargaron los 16 datasets y se calcularon métricas de calidad básicas.
- Se identificaron duplicados, nulos totales, y % de nulos en precio y área.
- Se visualizaron mapas de calor y gráficos de barras de nulos.
- Se analizaron patrones de nulos por ciudad y dataset.
- Se guardaron reportes en `outputs/`:
  - `calidad_grupo_a.csv`
  - `calidad_grupo_b.csv`
  - `reporte_nulos_completo.csv`
  - `mapa_calor_nulos_grupo_a.png`
  - `pct_nulos_por_columna_grupo_a.png`

**Hallazgos clave:**
- Los datasets A3 (House Prediction) y A8 (UPZ) presentan pocos nulos.
- A1 (Properati) tiene nulos significativos en área y ubicación.
- A2 (FincaRaiz) requiere limpieza por columnas con alta tasa de nulos.
- Los nulos en precio son bajos en la mayoría de datasets (< 5%).